In [1]:
import os
import pathlib
import shutil
import subprocess

In [2]:
readme_name   = 'README-Scotch'
scotch_path   = os.path.join(pathlib.Path.home(), 'repositories', 'Scotch')
temp_path     = '/tmp'
notebook_name = 'scotch-repository.ipynb'
notebook_path = os.path.join(pathlib.Path.home(), notebook_name) 

thorns = ['Whisky', 'Whisky_TOVSolverC', 'Whisky_IVP', 'Whisky_Init_Data', 'TracerHLLE', 'SoundSpeed', 'RiemannSolverROE', 'RiemannSolverMarquina', 'RiemannSolverHLLE', 'RiemannSolverHLL', 'RecoverMHD', 'RecoverMarti', 'ReconstructTVD', 'ReconstructPPM', 'ReconstructENO', 'QuasiGRMHD', 'HydroBase', 'EOS_Polytrope']

# Helper class for git repository

In [3]:
class git_repository:
    remote_format = 'git@github.gatech.edu:Numrel-code/{}.git'
    
    def __init__(self, repo_path):
        self.repo_path = repo_path
    
    def clone(self, remote_name):
        subprocess.run(['git', 'clone', git_repository.remote_format.format(remote_name), self.repo_path], check=True)
        
    def init(self):
        subprocess.run(['git', 'init'], check=True, cwd=self.repo_path)

    def add(self, file):
        subprocess.run(['git', 'add', file], check=True, cwd=self.repo_path)
    
    def commit(self, message):
        subprocess.run(['git', 'commit', '-m', message], check=True, cwd=self.repo_path)
        commit_hash = subprocess.run(['git', 'rev-parse', '--verify', 'HEAD'], check=True, cwd=self.repo_path, stdout=subprocess.PIPE).stdout.decode('utf-8').rstrip()
        return commit_hash
    
    def add_remote(self, repo_name, remote_name=None, fetch=False):
        if remote_name is None:
            remote_name = repo_name
            
        if not fetch:
            subprocess.run(['git', 'remote', 'add', remote_name, git_repository.remote_format.format(repo_name)], check=True, cwd=self.repo_path)
        else:
            subprocess.run(['git', 'remote', 'add', '-f', remote_name, git_repository.remote_format.format(repo_name)], check=True, cwd=self.repo_path)
    
    def list_branches(self):
        branches = []
        
        result = subprocess.run(['git', 'branch', '-a'], check=True, cwd=self.repo_path, stdout=subprocess.PIPE)    
        for line in result.stdout.decode('utf-8').splitlines():
            if 'remotes/origin' in line and '->' not in line:
                branches.append(line.split('/')[2])

        return branches
    
    def checkout_branch(self, branch):
        subprocess.run(['git', 'checkout', branch], check=True, cwd=self.repo_path)
    
    def make_branch(self, branch, start_hash=None):
        if start_hash is None:
            subprocess.run(['git', 'checkout', '-b', branch], check=True, cwd=self.repo_path)
        else:
            subprocess.run(['git', 'checkout', '-b', branch, start_hash], check=True, cwd=self.repo_path)
    
    def merge_remote(self, remote_name, branch):
        subprocess.run(['git', 'merge', '%s/%s'%(remote_name, branch), '--allow-unrelated-histories', '--no-edit'], check=True, cwd=self.repo_path)

    def push(self, remote, branch):
        subprocess.run(['git', 'push', remote, branch, '--force'], check=True, cwd=self.repo_path)

# Build command to move thorns into their own subdirectories

In [4]:
find_base = 'find -mindepth 1 -maxdepth 1 -not -name .git -and -not -name %s'%readme_name

for thorn in thorns:
    find_base += ' -and -not -name %s'%thorn

def move_command(thorn):
    return find_base + ' -exec git mv "{}" %s/ \;'%thorn

# Set up Scotch repository
- Make directory and initialize git repository.
- Write basic README file, add, and make initial commit. Save the initial commit hash to use as a starting point for all branches.
- Add origin remote, but don't push yet.

In [5]:
%%capture
if os.path.exists(scotch_path):
    shutil.rmtree(scotch_path)

os.makedirs(scotch_path)

scotch_repo = git_repository(scotch_path)
scotch_repo.init()

In [6]:
%%capture
with open(os.path.join(scotch_path, readme_name), 'w') as readme_file:
    readme_file.write('# Scotch GRMHD Code\n')
    readme_file.write('Details regarding consolidation of the original thorn repositories into this one can be found in the notebook `scotch-repository.ipynb`.')
    
scotch_repo.add(readme_name)
initial_commit_hash = scotch_repo.commit('Initial commit.')

In [7]:
%%capture
scotch_repo.add_remote('Scotch', remote_name='origin')

# Add remotes for each of the thorns

In [8]:
for thorn in thorns:
    scotch_repo.add_remote(thorn, fetch=True)

# Determine what branches are in each repository
- Make a list of all unique branches from all repositories.
- Make a list of the branches that common to every repository

In [9]:
branches = {}
branches_all = set()

for thorn in thorns:
    checkout_path = os.path.join(temp_path, thorn)
    repo = git_repository(checkout_path)
    repo.clone(thorn)

    branches[thorn] = repo.list_branches()
    branches_all |= set(branches[thorn])

    shutil.rmtree(checkout_path)

branches_common = set([branch for branch in branches_all if all(branch in branches[thorn] for thorn in thorns)])

# For all branches that are common to all thorns:
- Create the branch in the Scotch repository (unless it is master)
## For each thorn
    - Merge the thorn repository into the Scotch repository.
    - Move all of the thorn contents into a subdirectory for the thorn and commit changes.

In [10]:
print('Branches common to all thorns:')
for branch in branches_common:
    print('\t%s'%branch)
    scotch_repo.checkout_branch('master')
    if not branch == 'master':
        scotch_repo.make_branch(branch, start_hash=initial_commit_hash)
    
    for thorn in thorns:
        scotch_repo.merge_remote(thorn, branch)
        
        os.makedirs(os.path.join(scotch_path, thorn))
        subprocess.run(move_command(thorn), shell=True, check=True, cwd=scotch_path)
        
        scotch_repo.commit('Add %s thorn.'%thorn)

Branches common to all thorns:
	ET_2016_11
	master


# For all branches that are _not_ common to all thorns:
- Create the branch in the Scotch repository
## For each thorn
    - Merge the thorn repository into the Scotch repository. If the thorn repository does not contain this branch, merge in the master branch of the thorn.
    - Move all of the thorn contents into a subdirectory for the thorn and commit changes.

In [11]:
for branch in branches_all - branches_common:
    print('%s branch:'%branch)
    scotch_repo.checkout_branch('master')
    scotch_repo.make_branch(branch, start_hash=initial_commit_hash)
    
    for thorn in thorns:
        merge_branch = branch if branch in branches[thorn] else 'master'
        print('\tMerging in %s/%s'%(thorn, merge_branch))
        scotch_repo.merge_remote(thorn, merge_branch)
        
        os.makedirs(os.path.join(scotch_path, thorn))
        subprocess.run(move_command(thorn), shell=True, check=True, cwd=scotch_path)
        
        scotch_repo.commit('Add %s thorn.'%thorn)

develMHD branch:
	Merging in Whisky/develMHD
	Merging in Whisky_TOVSolverC/master
	Merging in Whisky_IVP/master
	Merging in Whisky_Init_Data/develMHD
	Merging in TracerHLLE/master
	Merging in SoundSpeed/master
	Merging in RiemannSolverROE/develMHD
	Merging in RiemannSolverMarquina/master
	Merging in RiemannSolverHLLE/master
	Merging in RiemannSolverHLL/master
	Merging in RecoverMHD/master
	Merging in RecoverMarti/master
	Merging in ReconstructTVD/develMHD
	Merging in ReconstructPPM/develMHD
	Merging in ReconstructENO/develMHD
	Merging in QuasiGRMHD/master
	Merging in HydroBase/master
	Merging in EOS_Polytrope/master
ET_2012_05 branch:
	Merging in Whisky/ET_2012_05
	Merging in Whisky_TOVSolverC/ET_2012_05
	Merging in Whisky_IVP/ET_2012_05
	Merging in Whisky_Init_Data/ET_2012_05
	Merging in TracerHLLE/ET_2012_05
	Merging in SoundSpeed/ET_2012_05
	Merging in RiemannSolverROE/ET_2012_05
	Merging in RiemannSolverMarquina/ET_2012_05
	Merging in RiemannSolverHLLE/ET_2012_05
	Merging in Rieman

# Make final changes to each branch of the new repository
- Return the README file to its proper name

In [ ]:
for branch in branches_all:
    scotch_repo.checkout_branch(branch)
    subprocess.run(['git', 'mv', readme_name, 'README.md'], check=True, cwd=scotch_path)
    scotch_repo.commit('Rename README.')
    
    shutil.copyfile(notebook_path, os.path.join(scotch_path, notebook_name))
    scotch_repo.add(notebook_name)
    scotch_repo.commit('Add this notebook.')
    
    scotch_repo.push('origin', branch)